# 07 Gaussian Naive Bayes With Basic Image Features

This notebook trains only `GaussianNB` using the existing basic image-derived features. Gaussian Naive Bayes is a probability-based supervised classifier. It assumes features are conditionally independent and approximately normally distributed within each class, which is probably weak for image features, so this is mainly a simple baseline.

## 1. Project setup

In [11]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/home/jp/MSE446_Nanoparticle_Ordering')

## 2. Imports

In [12]:
from src.extract_basic_features import BASIC_FEATURE_COLUMNS
from src.train_logistic_regression import FEATURES_CSV, load_features
from src.train_gaussian_nb import (
    SCORES_CSV,
    CONFUSION_MATRIX_FIGURE,
    build_comparison_table,
    train_gaussian_nb,
)

## 3. Paths and configuration

In [13]:
print(f"Basic features CSV: {FEATURES_CSV}")
print(f"Scores CSV: {SCORES_CSV}")
print(f"Confusion matrix figure: {CONFUSION_MATRIX_FIGURE}")

Basic features CSV: /home/jp/MSE446_Nanoparticle_Ordering/data/features_basic.csv
Scores CSV: /home/jp/MSE446_Nanoparticle_Ordering/results/model_scores_gaussian_nb_basic.csv
Confusion matrix figure: /home/jp/MSE446_Nanoparticle_Ordering/results/figures/confusion_matrix_gaussian_nb_basic.png


## 4. Load metadata

In [14]:
features = load_features(FEATURES_CSV)
features.head()

,filename,label,sample,area,area_group,kv,mm,mag,param_group,mean_intensity,...,min_intensity,max_intensity,p10_intensity,p25_intensity,p50_intensity,p75_intensity,p90_intensity,entropy,bright_pixel_ratio,edge_density
0,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.354890,...,0.160784,0.964706,0.266667,0.298039,0.345098,0.396078,0.447059,6.260059,0.099014,0.161224
1,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.414923,...,0.094118,1.000000,0.329412,0.376471,0.407843,0.447059,0.486275,6.191408,0.098190,0.113770
2,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.311404,...,0.141176,1.000000,0.247059,0.266667,0.286275,0.317647,0.423529,5.814114,0.097717,0.160583
3,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.326249,...,0.113725,1.000000,0.235294,0.266667,0.313726,0.368627,0.419608,6.261398,0.098694,0.164108
4,kv-10p0kV__mm-11p3mm__label-ordered__sample-S1...,ordered,S1,no_area,S1__no_area,10p0kV,11p3mm,100k,10p0kV__11p3mm__100k,0.260067,...,0.000000,1.000000,0.117647,0.176471,0.235294,0.294118,0.415686,6.846660,0.099258,0.171616


## 5. Sanity checks

In [15]:
print(f"Rows: {len(features)}")
print("Label counts:")
print(features["label"].value_counts().sort_index())
print(f"Image feature columns: {len(BASIC_FEATURE_COLUMNS)}")
print(BASIC_FEATURE_COLUMNS)
assert set(BASIC_FEATURE_COLUMNS).issubset(features.columns)
assert features["area_group"].notna().all()

Rows: 1000
Label counts:
label
disordered    250
ordered       750
Name: count, dtype: int64
Image feature columns: 12
['mean_intensity', 'std_intensity', 'min_intensity', 'max_intensity', 'p10_intensity', 'p25_intensity', 'p50_intensity', 'p75_intensity', 'p90_intensity', 'entropy', 'bright_pixel_ratio', 'edge_density']


## 6. Main analysis

Only image-derived feature columns are used as predictors. Metadata columns are used for labels, auditing, and grouped splitting, not as model inputs.

In [16]:
scores, report, matrix, figure_path, y_train, y_test = train_gaussian_nb(features)
scores

,model,train_accuracy,test_accuracy,train_balanced_accuracy,test_balanced_accuracy,train_macro_precision,macro_precision,train_macro_recall,macro_recall,train_macro_f1,macro_f1,disordered_recall,selected_split_seed,split_distribution_distance,train_size,test_size
0,gaussian_nb_basic,0.821596,0.790541,0.730829,0.698198,0.767458,0.719289,0.730829,0.698198,0.745446,0.70708,0.513514,140,0.0,852,148


## 7. Results/figures

Balanced accuracy and macro F1 matter more than raw accuracy because the dataset has many more ordered images than disordered images.

In [17]:
scores.to_csv(SCORES_CSV, index=False)

print("Selected split label counts:")
print("Train:")
print(y_train.value_counts().sort_index())
print("Test:")
print(y_test.value_counts().sort_index())
print("\nPer-class precision/recall/F1:")
print(report)
print("Confusion matrix rows=true, columns=predicted:")
print(matrix)
print("\nModel comparison:")
print(build_comparison_table(scores))
print(f"\nSaved scores to {SCORES_CSV}")
print(f"Saved confusion matrix figure to {figure_path}")

Selected split label counts:
Train:
label
disordered    213
ordered       639
Name: count, dtype: int64
Test:
label
disordered     37
ordered       111
Name: count, dtype: int64

Per-class precision/recall/F1:
              precision    recall  f1-score   support

  disordered       0.59      0.51      0.55        37
     ordered       0.84      0.88      0.86       111

    accuracy                           0.79       148
   macro avg       0.72      0.70      0.71       148
weighted avg       0.78      0.79      0.79       148

Confusion matrix rows=true, columns=predicted:
            disordered  ordered
disordered          19       18
ordered             13       98

Model comparison:
                model  test_accuracy  test_balanced_accuracy  macro_f1  \
0     DummyClassifier       0.885787                0.500000  0.469717   
1  LogisticRegression       0.790541                0.779279  0.747676   
2  DecisionTree tuned       0.858108                0.824324  0.815691   
3  Ra

## 8. Notes for report

- GaussianNB is a probability-based classifier.
- It assumes feature independence and normal numeric feature distributions within each class.
- This assumption is probably weak for image features, so GaussianNB is mainly a simple baseline.
- Compare it using macro F1 and balanced accuracy, not raw accuracy, because the dataset is imbalanced.
- This step does not add SVC, KNN, HOG, graph features, augmentation, or CNNs.